# SILVER (Dados Limpos) 
Dados validados e limpos<br>
Tipos de dados corretos<br>
Valores nulos tratados<br>
Duplicatas removidas<br>

## PROCESSAMENTO DOS DADOS

### IMPORTAÇÃO DAS BIBLIOTECAS

In [6]:
from pathlib import Path
from pyspark.sql import functions as F
from spark_utils import get_spark, write_single_csv

spark = get_spark("SilverLayer")


### CARREGAMENTO DOS DADOS

In [7]:
bronze_file = Path("data/bronze/dados_brutos.csv")
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(bronze_file))
    .cache()
)
print(f"Dados carregados da camada Bronze ({df.count()} registros).")


Dados carregados da camada Bronze (10476 registros).


## TRATAMENTO DOS DADOS


### ALTERAÇÃO DOS TIPOS DE DADOS

In [8]:
df = df.withColumn("CODIGO_CLIENTE", F.col("CODIGO_CLIENTE").cast("string"))

categorical_cols = ["UF", "ESCOLARIDADE", "ESTADO_CIVIL"]
for col in categorical_cols:
    df = df.withColumn(col, F.upper(F.trim(F.col(col))))

bool_cols = ["CASA_PROPRIA", "OUTRA_RENDA", "TRABALHANDO_ATUALMENTE"]
for col in bool_cols:
    cleaned = F.upper(F.trim(F.col(col)))
    df = df.withColumn(
        col,
        F.when(cleaned == F.lit("SIM"), F.lit(True))
         .when(cleaned.isin("NAO", "NÃO"), F.lit(False))
         .otherwise(None)
    )

int_cols = ["IDADE", "QT_FILHOS", "QT_IMOVEIS", "TEMPO_ULTIMO_EMPREGO_MESES", "QT_CARROS", "SCORE"]
for col in int_cols:
    df = df.withColumn(col, F.col(col).cast("int"))

float_cols = ["VL_IMOVEIS", "OUTRA_RENDA_VALOR", "ULTIMO_SALARIO", "VALOR_TABELA_CARROS"]
for col in float_cols:
    df = df.withColumn(col, F.col(col).cast("double"))


### PADRONIZAÇÃO DOS VALORES TEXTUAIS 

In [9]:
string_cols = [name for name, dtype in df.dtypes if dtype == "string"]
for col in string_cols:
    df = df.withColumn(col, F.upper(F.trim(F.col(col))))
print(f"Colunas textuais padronizadas: {string_cols}")


Colunas textuais padronizadas: ['CODIGO_CLIENTE', 'UF', 'ESCOLARIDADE', 'ESTADO_CIVIL', 'ARQUIVO_FONTE']


### TRATAMENTO DE NULOS

In [10]:
null_counts = df.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df.columns])
null_counts.show(truncate=False)


+--------------+---+-----+------------+------------+---------+------------+----------+----------+-----------+-----------------+--------------------------+----------------------+--------------+---------+-------------------+-----+-----------+-------------+
|CODIGO_CLIENTE|UF |IDADE|ESCOLARIDADE|ESTADO_CIVIL|QT_FILHOS|CASA_PROPRIA|QT_IMOVEIS|VL_IMOVEIS|OUTRA_RENDA|OUTRA_RENDA_VALOR|TEMPO_ULTIMO_EMPREGO_MESES|TRABALHANDO_ATUALMENTE|ULTIMO_SALARIO|QT_CARROS|VALOR_TABELA_CARROS|SCORE|DATA_UPLOAD|ARQUIVO_FONTE|
+--------------+---+-----+------------+------------+---------+------------+----------+----------+-----------+-----------------+--------------------------+----------------------+--------------+---------+-------------------+-----+-----------+-------------+
|0             |0  |0    |0           |0           |0        |0           |0         |0         |0          |0                |0                         |0                     |3             |0        |0                  |0    |0      

In [11]:
df = df.replace('SEM DADOS', None)
median_result = df.approxQuantile("ULTIMO_SALARIO", [0.5], 0.01)
median_sal = median_result[0] if median_result else None
if median_sal is not None:
    df = df.withColumn(
        "ULTIMO_SALARIO",
        F.when(F.col("ULTIMO_SALARIO").isNull(), F.lit(median_sal)).otherwise(F.col("ULTIMO_SALARIO"))
    )
print(f"Mediana utilizada para ULTIMO_SALARIO: {median_sal}")


Mediana utilizada para ULTIMO_SALARIO: 6100.0


### TRATAMENTO DE OUTLIERS


In [12]:
qt_mode_row = (
    df.groupBy("QT_FILHOS")
      .count()
      .orderBy(F.desc("count"))
      .first()
)
qt_moda = qt_mode_row["QT_FILHOS"] if qt_mode_row else 0
df = df.withColumn(
    "QT_FILHOS",
    F.when(F.col("QT_FILHOS") > 3, F.lit(qt_moda)).otherwise(F.col("QT_FILHOS"))
)
print(f"Moda aplicada para QT_FILHOS: {qt_moda}")


Moda aplicada para QT_FILHOS: 1


### TRATAMENTO DE DUPLICATAS

In [13]:
total_registros = df.count()
df_distinct = df.dropDuplicates()
duplicados = total_registros - df_distinct.count()
print(f"Duplicatas removidas: {duplicados}")
df = df_distinct
df.printSchema()


Duplicatas removidas: 0
root
 |-- CODIGO_CLIENTE: string (nullable = true)
 |-- UF: string (nullable = true)
 |-- IDADE: integer (nullable = true)
 |-- ESCOLARIDADE: string (nullable = true)
 |-- ESTADO_CIVIL: string (nullable = true)
 |-- QT_FILHOS: integer (nullable = true)
 |-- CASA_PROPRIA: boolean (nullable = true)
 |-- QT_IMOVEIS: integer (nullable = true)
 |-- VL_IMOVEIS: double (nullable = true)
 |-- OUTRA_RENDA: boolean (nullable = true)
 |-- OUTRA_RENDA_VALOR: double (nullable = true)
 |-- TEMPO_ULTIMO_EMPREGO_MESES: integer (nullable = true)
 |-- TRABALHANDO_ATUALMENTE: boolean (nullable = true)
 |-- ULTIMO_SALARIO: double (nullable = true)
 |-- QT_CARROS: integer (nullable = true)
 |-- VALOR_TABELA_CARROS: double (nullable = true)
 |-- SCORE: integer (nullable = true)
 |-- DATA_UPLOAD: timestamp (nullable = true)
 |-- ARQUIVO_FONTE: string (nullable = true)



### CRIAÇÃO DE COLUNA RENDA_TOTAL

In [14]:
df = df.withColumn(
    "RENDA_TOTAL",
    F.coalesce(F.col("ULTIMO_SALARIO"), F.lit(0.0)) + F.coalesce(F.col("OUTRA_RENDA_VALOR"), F.lit(0.0))
)
df.select("RENDA_TOTAL").show(5)


+-----------+
|RENDA_TOTAL|
+-----------+
|     9800.0|
|     3900.0|
|    17000.0|
|     4800.0|
|    17500.0|
+-----------+
only showing top 5 rows


## SALVAR NA CAMADA SILVER

### ADICIONAR INFORMAÇÃO DE TRATAMENTO DOS DADOS
 

In [15]:
df = df.withColumn("DATA_TRATAMENTO", F.current_timestamp())


In [16]:
silver_path = "data/silver/dados_limpos.csv"
write_single_csv(df, silver_path)
print(f"Dados limpos salvos: {silver_path} (registros={df.count()})")
df.show(5, truncate=False)


Dados limpos salvos: data/silver/dados_limpos.csv (registros=10476)
+--------------+---+-----+---------------------+------------+---------+------------+----------+----------+-----------+-----------------+--------------------------+----------------------+--------------+---------+-------------------+-----+-----------------------+------------------+-----------+-------------------------+
|CODIGO_CLIENTE|UF |IDADE|ESCOLARIDADE         |ESTADO_CIVIL|QT_FILHOS|CASA_PROPRIA|QT_IMOVEIS|VL_IMOVEIS|OUTRA_RENDA|OUTRA_RENDA_VALOR|TEMPO_ULTIMO_EMPREGO_MESES|TRABALHANDO_ATUALMENTE|ULTIMO_SALARIO|QT_CARROS|VALOR_TABELA_CARROS|SCORE|DATA_UPLOAD            |ARQUIVO_FONTE     |RENDA_TOTAL|DATA_TRATAMENTO          |
+--------------+---+-----+---------------------+------------+---------+------------+----------+----------+-----------+-----------------+--------------------------+----------------------+--------------+---------+-------------------+-----+-----------------------+------------------+-----------+--